# 07 — Unattended 2-model campaign (disconnect-proof)

Trains **two large (YOLO26l) models on all GPUs**, each through the full pipeline
(real batch probe -> coarse train until early stop -> fine-tune the best checkpoint ->
evaluate on test -> log per-class best confidence):

1. `campaign_l_1280` - 1280x1280 tiles, uncompressed
2. `campaign_l_640`  - the same tiles down-scaled ("compressed") to 640x640

It is launched **detached with `nohup`** so it keeps running even if your connection
drops. Every stage records state to disk, so re-running the launch cell **resumes** from
where it stopped. Progress goes to `runs/campaign.log`.

## Launch (detached, self-restarting). Safe to re-run: it resumes, never restarts from scratch.

Launches `supervise_campaign.sh`, which relaunches `run_campaign.py` until both models
are fully done. Resilience is layered:
- **per stage**: coarse / fine-tune / eval / per-class-conf are checkpointed and skipped if done;
- **per training**: a crashed coarse/fine-tune **resumes from its last checkpoint**;
- **per run**: each model is retried up to 8×, and the campaign **always moves on to the 2nd model**;
- **per process**: the supervisor relaunches the whole thing if the process itself is killed.

In [ ]:
import subprocess
ROOT = '/home/jovyan/shared/s0598584'
# self-restarting supervisor, fully detached (survives disconnects)
cmd = (f"cd {ROOT} && chmod +x scripts/supervise_campaign.sh && "
       f"nohup setsid bash scripts/supervise_campaign.sh > runs/campaign.log 2>&1 < /dev/null &")
running = subprocess.run('pgrep -af "[s]upervise_campaign|[r]un_campaign.py"',
                         shell=True, capture_output=True, text=True).stdout.strip()
if running:
    print('campaign already running:\n', running)
else:
    subprocess.run(cmd, shell=True, executable='/bin/bash')
    print('campaign launched (supervised, detached) -> runs/campaign.log')

## Monitor progress (re-run anytime)

In [ ]:
import subprocess, os
ROOT = '/home/jovyan/shared/s0598584'
print('--- alive? ---')
print(subprocess.run('pgrep -af "[s]upervise_campaign|[r]un_campaign.py|[_]temp_" || echo "not running"',
                     shell=True, capture_output=True, text=True).stdout)
print('--- campaign_state.json ---')
sp = f'{ROOT}/runs/campaign_state.json'
print(open(sp).read() if os.path.exists(sp) else '(not created yet)')
print('--- last 25 log lines ---')
print(subprocess.run(f'tail -n 25 {ROOT}/runs/campaign.log', shell=True, capture_output=True, text=True).stdout)

## Results: per-class best confidence (combined test+val), both runs

In [ ]:
import os
ROOT = '/home/jovyan/shared/s0598584'
csv = f'{ROOT}/runs/best_conf_log.csv'
print(open(csv).read() if os.path.exists(csv) else '(no results yet)')
for name in ['campaign_l_1280', 'campaign_l_640']:
    rep = f'{ROOT}/runs/faces/{name}/eval_report.md'
    if os.path.exists(rep):
        print('\n==== ', name, ' ====')
        print(open(rep).read())